# Fine-tune Parakeet CTC for Italian ASR

Fine-tune [NVIDIA Parakeet CTC 0.6B](https://huggingface.co/nvidia/parakeet-ctc-0.6b) on Italian speech for the **Anti-Vocale** app.

Based on the [smol-audio](https://github.com/Deep-unlearning/smol-audio) Parakeet notebook.

## Before you start
1. **Runtime > Change runtime type > T4 GPU > Save**
2. **Runtime > Run all** (or run cells top to bottom with Shift+Enter)
3. Do NOT skip cells. Do NOT run extra pip installs.

## Modes
- **Smoke test** (default): 500 samples, 100 steps, ~15 min
- **Real run**: set `SMOKE_TEST = False` in cell 3, ~2-3 hours

In [ ]:
!pip install -q transformers "datasets<3.0" accelerate peft librosa soundfile jiwer --upgrade torchao

## Load Dataset

We use [Google Fleurs Italian](https://huggingface.co/datasets/google/fleurs) (`it_it` config). This is a public dataset with ~3.4 hours of Italian speech — no authentication needed.

The dataset has an `audio` column (resampled to 16kHz) and a `transcription` column (renamed to `text` to match the data collator's expectations).

In [ ]:
from datasets import load_dataset, Audio

SMOKE_TEST = True  # Set to False for real run
NUM_SAMPLES = 500 if SMOKE_TEST else 5000
MAX_AUDIO_SEC = 20  # Filter out samples longer than 20s to avoid OOM

dataset = load_dataset(
    "google/fleurs",
    "it_it",
    split="train",
    streaming=True,
    trust_remote_code=True,
)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Filter out long audio samples
def is_short(sample):
    return len(sample["audio"]["array"]) <= MAX_AUDIO_SEC * 16000

dataset = dataset.filter(is_short).take(NUM_SAMPLES)

# Rename to match the data collator's expected "text" column
dataset = dataset.rename_column("transcription", "text")

print(f"Dataset ready (streaming {NUM_SAMPLES} samples, max {MAX_AUDIO_SEC}s each)")
sample = next(iter(dataset))
print(f"Sample keys: {list(sample.keys())}")
print(f"Duration: {len(sample['audio']['array']) / 16000:.1f}s")
print(f"Text: {sample['text'][:80]}...")

# Reload (peeking consumed one sample)
dataset = load_dataset(
    "google/fleurs",
    "it_it",
    split="train",
    streaming=True,
    trust_remote_code=True,
)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
dataset = dataset.filter(is_short).take(NUM_SAMPLES)
dataset = dataset.rename_column("transcription", "text")

## Load Model and Processor

In [ ]:
import torch
from transformers import AutoModelForCTC, AutoProcessor

model_checkpoint = "nvidia/parakeet-ctc-0.6b"

processor = AutoProcessor.from_pretrained(model_checkpoint)
model = AutoModelForCTC.from_pretrained(model_checkpoint)
model = model.float()  # Force all params+buffers to float32 (conv biases load as bfloat16)

print(f"Model: {model_checkpoint}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Data Collator

The data collator processes raw audio and text through the Parakeet processor, which handles feature extraction and label tokenization for CTC training.

In [ ]:
class ParakeetDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        texts = [f["text"] for f in features]
        audios = [f["audio"]["array"] for f in features]

        inputs = self.processor(
            audio=audios,
            text=texts,
            sampling_rate=self.processor.feature_extractor.sampling_rate,
        )
        return inputs

data_collator = ParakeetDataCollator(processor)

## LoRA Fine-tuning

Apply LoRA to the model's linear layers for parameter-efficient training. This keeps memory usage low on the T4 GPU.

In [ ]:
from peft import LoraConfig, get_peft_model

model = AutoModelForCTC.from_pretrained(model_checkpoint)
model = model.float()  # Force all params+buffers to float32

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["linear"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer

MAX_STEPS = 100 if SMOKE_TEST else 1000

training_args = TrainingArguments(
    output_dir="./parakeet-ctc-0.6b-italian-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    max_grad_norm=1.0,
    max_steps=MAX_STEPS,
    warmup_steps=10 if SMOKE_TEST else 100,
    bf16=False,
    fp16=False,
    logging_steps=10,
    save_steps=50 if SMOKE_TEST else 200,
    eval_strategy="no",
    save_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

print(f"Starting LoRA training for {MAX_STEPS} steps...")
trainer.train()
print("Training complete!")

## Inference

Test the fine-tuned model on an Italian sample from Fleurs.

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    feature_extractor=processor.feature_extractor,
    tokenizer=processor.tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

# Get a test sample from Fleurs Italian
test_ds = load_dataset(
    "google/fleurs", "it_it",
    split="train", streaming=True, trust_remote_code=True,
)
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))
test_sample = next(iter(test_ds))

result = pipe(test_sample["audio"]["array"])
print(f"Ground truth: {test_sample['transcription']}")
print(f"Prediction:   {result['text']}")

In [ ]:
# Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/parakeet-ctc-0.6b-italian-lora"
model.save_pretrained(save_path)
processor.save_pretrained(save_path)
print(f"Saved to {save_path}")

## Next steps

```python
# Merge LoRA weights
from peft import AutoPeftModelForCTC
merged = AutoPeftModelForCTC.from_pretrained(save_path)
merged = merged.merge_and_unload()
merged.save_pretrained("./parakeet-ctc-0.6b-italian-merged")
```

```bash
# ONNX export (locally)
python -m transformers.onnx --model=./parakeet-ctc-0.6b-italian-merged --feature=ctc ./onnx-output
```